# Chapter 9 Companion: Complex Coefficients from One Unit Cell

**How do a prescribed transmission and its spatial samples determine the complex coefficients of a periodic exit field?**

[7.4](../7_4_Diffraction_Kinematics_and_Complex_Order_Coefficients.ipynb) explains the physical meaning of the coefficients. [9.2](../9_2_Sampling_Complex_Fields_and_Their_Spectra.ipynb) studies how sampling preserves them. This computational companion develops the thin-mask coefficient calculations for a rectangular opening, a nonseparable cell, and a lateral translation.

We use normal-incidence plane-wave illumination with $A_{\mathrm{inc}}=1$ and zero incident phase at the ideal exit plane $z=0^-$. Hence

$$U_{\mathrm{exit}}(x,y)=t(x,y)U_i(x,y)=t(x,y),\qquad C_{mn}=t_{mn}.$$

One unit cell repeats across the ideal grating. Its periods are $p_x=4$ and $p_y=5$ micrometres, with rectangular opening fractions $(D_x,D_y)=(1/2,3/8)$. All transforms use fractional coordinates $x/p_x,y/p_y$. The coefficient calculation needs no wavelength or objective; wavelength and exterior medium enter when classifying the resulting harmonics as propagating or evanescent.

The period called $d$ in 7.4 and 9.1 is called $p$ in 9.2. Here $p_x,p_y$ name the two periods, and the output-plane field is written consistently as $U_{\mathrm{exit}}$.

Run this notebook independently from a fresh kernel. These examples prescribe the scalar thin-mask field; the physical scalar FMM comparison belongs to the exit-field modeling lesson, 9.3.

**Notation:** use $U_{\mathrm{exit}}$ at the exit plane, $U_{\mathrm{ref}}$ at a general reference plane, and $C_m$ (or $C_{mn}$) for field coefficients. [Shared convention](../0_0_Foundations_of_Scalar_Optics.ipynb#exit-field-notation).


In [ ]:
%matplotlib inline
from pathlib import Path
import sys, importlib
_support_dir = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                     if (p/'wave_optics'/'__init__.py').is_file()), None)
if _support_dir is None:
    raise FileNotFoundError('Open within the Grating_Physics folder.')
if str(_support_dir) not in sys.path:
    sys.path.insert(0, str(_support_dir))
from wave_optics import lattice_grating as lg
importlib.reload(lg)
import numpy as np
import matplotlib.pyplot as plt


px,py,Dx,Dy = 4.,5.,.5,.375
v=np.arange(-6,7)
mm,nn=np.meshgrid(v,v)
labels=np.column_stack([mm.ravel(),nn.ravel()])
coeff=lg.rectangular_coefficients(labels,(Dx,Dy))

<a id="two-dimensional-thin-mask-coefficients"></a>

## 1. Rectangular-opening coefficients

Prescribe $U_{\mathrm{exit}}(x,y)=t(x,y)U_i(x,y)$ at $z=0^-$. Orthogonality of the Fourier basis gives
$$t_{mn}=\frac1{p_xp_y}\int_{\rm cell}t(x,y)
 e^{-i2\pi(mx/p_x+ny/p_y)}\,dx\,dy.$$
For a centered rectangular opening, the two integrals factor. Each is
$$\frac1p\int_{-Dp/2}^{Dp/2}e^{-i2\pi mx/p}\,dx=D\operatorname{sinc}(mD),$$
so
$$\boxed{t_{mn}=D_xD_y\operatorname{sinc}(mD_x)\operatorname{sinc}(nD_y)},\qquad
\operatorname{sinc}(u)=\frac{\sin\pi u}{\pi u}.$$
At the reference plane $z=0^-$, $C_{mn}=A_{\rm inc}t_{mn}$. We use $A_{\rm inc}=1$ with phase zero at the origin. This prescribed thin-mask calculation **does not solve material scattering**, Fresnel reflection, polarization, thickness, or coupling inside a physical grating.

An allowed location may have zero amplitude. Squared magnitudes $|t_{mn}|^2$ are coefficient weights, distinct from normal-flux efficiencies. For a scalar model whose flux is proportional to $\operatorname{Im}(U^*\partial_zU)$, a propagating transmitted order has relative normal flux $(-\operatorname{Re}k_{z,mn})/(-k_{z,\rm inc})\,|C_{mn}/A_{\rm inc}|^2$. Other field definitions can require impedance factors. A prescribed mask is not a solved energy balance; grazing and evanescent single orders carry no outgoing normal flux in this scalar convention.

Moving the opening by $\Delta\mathbf r$ at fixed coordinates multiplies its coefficient by $e^{-i(m\mathbf b_1+n\mathbf b_2)\cdot\Delta\mathbf r}$. Changing the coordinate origin instead also changes incident-reference phases: coefficient phases must always state their origin.

### Recover the same coefficients from unit-cell samples

Use endpoint-excluded samples $x_j/p_x=x_0+j/N_x$, $y_l/p_y=y_0+l/N_y$. The normalized FFT is
$$t^{\rm num}_{mn}=\frac{e^{-i2\pi(mx_0+ny_0)}}{N_xN_y}
\sum_{j,l}t_{lj}e^{-i2\pi(mj/N_x+nl/N_y)}.$$
The origin factor matters. Midpoint sampling avoids jumps on sample points here; finite quadrature error remains. Never transform intensity in place of complex transmission.


In [ ]:
errors = []
for N in [128, 256, 512]:
    u = -.5+(np.arange(N)+.5)/N
    sample = ((abs(u[None,:]) < Dx/2) & (abs(u[:,None]) < Dy/2)).astype(float)
    numeric = lg.sampled_coefficients(sample, labels, origin=(u[0],u[0]))
    errors.append(np.max(abs(numeric-coeff)))
print('Maximum coefficient errors for N=128,256,512:', errors)
assert errors[-1] < 4e-5 and errors[-1] < errors[0]/10

fig, axes = plt.subplots(1,3, figsize=(12,3.6), constrained_layout=True)
shape = mm.shape
phase = np.ma.masked_where(abs(coeff.reshape(shape)) < 1e-10, np.angle(coeff.reshape(shape)))
for ax, image, title, cmap in zip(axes,
        [abs(coeff).reshape(shape), phase, abs(numeric-coeff).reshape(shape)],
        ['Analytic magnitude', 'Phase (rad); zeros masked', 'Absolute FFT error'],
        ['viridis', 'twilight', 'magma']):
    im = ax.imshow(image, origin='lower', extent=[-6.5,6.5,-6.5,6.5], cmap=cmap,
                   **(dict(vmin=-np.pi,vmax=np.pi) if cmap=='twilight' else {}))
    ax.set(title=title, xlabel='m', ylabel='n')
    fig.colorbar(im, ax=ax, shrink=.75)
plt.show()


## 2. Recover the one-dimensional limit

If $t$ is independent of $y$, integration over $y$ gives $\delta_{n0}$. Choosing $D_y=1$ recovers $t_{mn}=D_x\operatorname{sinc}(mD_x)\delta_{n0}$. All off-row locations still exist as candidate labels of this redundant two-period description, but they are unexcited. Conical incidence shifts the populated row to $k_y=k_{y,\rm inc}$.

## 3. Compute a nonseparable cell

Use two disjoint rectangles with different aspect ratios and centers in fractional cell coordinates. Their union is not a product $f(x)g(y)$: the available $y$ intervals depend on $x$. Its coefficient is the sum of two shifted rectangle coefficients. This also produces phases beyond 0 and $\pi$.


In [ ]:
parts = [((.25,.375), (-.25,-.1875)), ((.375,.25), (.1875,.25))]
nonsep = sum(lg.rectangular_coefficients(labels, duty, shift) for duty, shift in parts)
N = 512
u = -.5+(np.arange(N)+.5)/N
cell = np.zeros((N,N))
for duty, shift in parts:
    cell += ((abs(u[None,:]-shift[0]) < duty[0]/2) &
             (abs(u[:,None]-shift[1]) < duty[1]/2))
assert cell.max() == 1  # disjoint: sum equals union
nonsep_numeric = lg.sampled_coefficients(cell, labels, origin=(u[0],u[0]))
assert np.max(abs(nonsep_numeric-nonsep)) < 5e-5
singular = np.linalg.svd(nonsep.reshape(mm.shape), compute_uv=False)
assert singular[1] > .01  # coefficient matrix is not rank one
fig, axes = plt.subplots(1,3, figsize=(11,3.5), constrained_layout=True)
axes[0].imshow(cell, origin='lower', extent=[-.5,.5,-.5,.5], cmap='gray', vmin=0,vmax=1)
axes[0].set(title='Nonseparable clear cell', xlabel='x/px', ylabel='y/py')
for ax, arr, title, cmap in [(axes[1], abs(nonsep).reshape(mm.shape), 'Magnitude', 'viridis'),
        (axes[2], np.ma.masked_where(abs(nonsep.reshape(mm.shape))<1e-10,
                                    np.angle(nonsep.reshape(mm.shape))), 'Phase (rad)', 'twilight')]:
    im=ax.imshow(arr, origin='lower', extent=[-6.5,6.5,-6.5,6.5], cmap=cmap,
                 **(dict(vmin=-np.pi,vmax=np.pi) if cmap=='twilight' else {}))
    ax.set(title=title, xlabel='m', ylabel='n')
    fig.colorbar(im, ax=ax, shrink=.75)
plt.show()


## 4. Check translation and the coefficient reference

A displaced opening changes phases at a fixed laboratory origin. If the new coordinates instead use an origin at $\mathbf r_0$, the full surface coefficient transforms as $C'_{mn}=C_{mn}e^{i\mathbf k_{\parallel,mn}\cdot\mathbf r_0}$; the mask coefficient uses only the reciprocal increment, and the incident amplitude gains its incident phase. Thus origin changes and physical translations are distinct operations.

At $z=0^-$ the coefficients above are normalized to $A_{\rm inc}=1$ with zero phase at the laboratory origin. At $z=-s$ they acquire $e^{-ik_{z,mn}s}$. The following checks compare the analytic 1D limit and an exact sample roll with the Fourier translation theorem. The numerical coefficient errors are quadrature errors, not material-scattering errors.


In [ ]:
one_d=lg.rectangular_coefficients(labels,(Dx,1.))
np.testing.assert_allclose(one_d[labels[:,1]!=0],0,atol=1e-15)
np.testing.assert_allclose(one_d[labels[:,1]==0],Dx*np.sinc(labels[labels[:,1]==0,0]*Dx))
base_cell=((abs(u[None,:])<Dx/2)&(abs(u[:,None])<Dy/2)).astype(float)
shift_pixels=(37,-21)
shifted=np.roll(base_cell,(shift_pixels[1],shift_pixels[0]),axis=(0,1))
c0=lg.sampled_coefficients(base_cell,labels,origin=(u[0],u[0]))
c1=lg.sampled_coefficients(shifted,labels,origin=(u[0],u[0]))
phase_shift=np.exp(-2j*np.pi*(labels@np.array(shift_pixels)/N))
np.testing.assert_allclose(c1,c0*phase_shift,atol=1e-14)
analytic_shift=lg.rectangular_coefficients(labels,(Dx,Dy),np.array(shift_pixels)/N)
np.testing.assert_allclose(analytic_shift,coeff*phase_shift,atol=1e-15)
print('1D excitation limit and complex lateral-translation checks passed.')


## What the checks establish

The sampled complex transmission converges to the analytical coefficients, the one-dimensional limit excites only the $n=0$ row, and a lateral translation produces the expected coefficient-phase change. These verify representation of the prescribed thin-mask field. They do not validate a material-scattering model.

The FFT samples cover one complete cell and exclude repeated endpoints. Read [9.2](../9_2_Sampling_Complex_Fields_and_Their_Spectra.ipynb) for sampling and aliasing, and [Appendix 4](../Appendix_4_How_to_Use_FFTs_in_Python.ipynb) for Fourier-series/FT normalization and NumPy conventions.

**Physical interpretation:** [7.4: complex order coefficients](../7_4_Diffraction_Kinematics_and_Complex_Order_Coefficients.ipynb). **Order locations:** [5.3: two-dimensional order lattices](../5_3_Two_Dimensional_Gratings_and_Diffraction_Order_Lattices.ipynb). **Chapter overview:** [9.0](../9_0_From_a_Physical_Model_to_a_Numerical_Experiment.ipynb).